#  Virtual Air Drawing Board

**Use your finger as a pen in front of the webcam!**

---

##  Features
-  Raise **1 finger** → Draw on screen
-  Raise **2 fingers** → Select color from buttons
-  Colors: **Blue, Green, Red, Eraser**
- # Press **`C`** to clear canvas | Press **`Q`** to quit

## Tech Stack
| Library | Purpose |
|---|---|
| `opencv-python` | Webcam capture & drawing |
| `mediapipe` | Hand landmark detection |
| `numpy` | Canvas creation |


##  Step 1 — Install Dependencies

In [1]:
# Install required libraries
!pip install opencv-python mediapipe numpy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\HP\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


##  Step 2 — Download the Hand Landmark Model

MediaPipe 0.10.x uses a **Tasks API** that requires a `.task` model file.
We download it once and reuse it.

In [2]:
import urllib.request
import os

MODEL_PATH = "hand_landmarker.task"

if not os.path.exists(MODEL_PATH):
    print("Downloading hand landmark model (~8MB)...")
    url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
    urllib.request.urlretrieve(url, MODEL_PATH)
    print(" Model downloaded successfully!")
else:
    print(" Model already exists, skipping download.")

 Model already exists, skipping download.


## 📚 Step 3 — Import Libraries

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

print("All libraries imported successfully!")
print(f"   OpenCV  version : {cv2.__version__}")
print(f"   NumPy   version : {np.__version__}")
print(f"   MediaPipe version: {mp.__version__}")

 All libraries imported successfully!
   OpenCV  version : 4.13.0
   NumPy   version : 2.4.1
   MediaPipe version: 0.10.32


##  Step 4 — Define Hand Connections

MediaPipe's 21 hand landmarks connected to draw the skeleton manually.

In [ ]:
# 21 hand landmark connections (pairs of landmark indices)
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),        # Thumb
    (0,5),(5,6),(6,7),(7,8),        # Index finger
    (0,9),(9,10),(10,11),(11,12),   # Middle finger
    (0,13),(13,14),(14,15),(15,16), # Ring finger
    (0,17),(17,18),(18,19),(19,20), # Pinky finger
    (5,9),(9,13),(13,17)            # Palm connections
]

print(f" {len(HAND_CONNECTIONS)} hand connections defined.")

 23 hand connections defined.


##  Step 5 — Define Color Buttons

Each button has a label, screen coordinates `(x1, y1, x2, y2)`, and a BGR color.

In [ ]:
# Button definitions: (Label, (x1, y1, x2, y2), BGR_color)
buttons = [
    ("BLUE",   (50,  20, 170, 70), (255, 0,   0  )),
    ("GREEN",  (190, 20, 310, 70), (0,   200, 0  )),
    ("RED",    (330, 20, 450, 70), (0,   0,   255)),
    ("ERASER", (470, 20, 620, 70), (0,   0,   0  )),
]

print(" Color buttons defined:")
for name, coords, color in buttons:
    print(f"   {name:8s} → coords={coords}, BGR={color}")

 Color buttons defined:
   BLUE     → coords=(50, 20, 170, 70), BGR=(255, 0, 0)
   GREEN    → coords=(190, 20, 310, 70), BGR=(0, 200, 0)
   RED      → coords=(330, 20, 450, 70), BGR=(0, 0, 255)
   ERASER   → coords=(470, 20, 620, 70), BGR=(0, 0, 0)


##  Step 6 — Helper Functions

In [6]:
def fingers_up(landmarks, h, w):
    """
    Detect if index and middle fingers are raised.
    Compares fingertip Y position vs PIP joint Y position.
    Returns: [index_up (bool), middle_up (bool)]
    """
    tips = [8, 12]   # Index tip, Middle tip
    pips = [6, 10]   # Index PIP, Middle PIP
    return [landmarks[t].y < landmarks[p].y for t, p in zip(tips, pips)]


def draw_skeleton(frame, landmarks, h, w):
    """
    Draw hand skeleton (bones + joints) on frame using OpenCV.
    """
    # Convert normalized coords to pixel coords
    pts = [(int(lm.x * w), int(lm.y * h)) for lm in landmarks]

    # Draw bones (connections)
    for a, b in HAND_CONNECTIONS:
        cv2.line(frame, pts[a], pts[b], (0, 220, 120), 2)

    # Draw joints (landmark dots)
    for i, pt in enumerate(pts):
        radius = 6 if i in [4, 8, 12, 16, 20] else 3  # Fingertips are bigger
        cv2.circle(frame, pt, radius, (255, 255, 255), cv2.FILLED)
        cv2.circle(frame, pt, radius, (0, 180, 100), 1)


def draw_buttons(frame, draw_color):
    """
    Draw color selection buttons at the top of the frame.
    Highlights the currently active color.
    """
    for name, (x1, y1, x2, y2), color in buttons:
        is_eraser = name == "ERASER"
        is_active = (draw_color == (0,0,0) and is_eraser) or \
                    (not is_eraser and draw_color == color)

        btn_color = (240, 240, 240) if is_eraser else color
        txt_color = (0, 0, 0)       if is_eraser else (255, 255, 255)

        # Drop shadow
        cv2.rectangle(frame, (x1+3, y1+3), (x2+3, y2+3), (20, 20, 20), cv2.FILLED)
        # Button fill
        cv2.rectangle(frame, (x1, y1), (x2, y2), btn_color, cv2.FILLED)
        # Active highlight border
        if is_active:
            cv2.rectangle(frame, (x1-3, y1-3), (x2+3, y2+3), (255, 255, 255), 3)
        # Label
        cv2.putText(frame, name, (x1+10, y1+32),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, txt_color, 2)


print(" Helper functions defined!")

✅ Helper functions defined!


##  Step 7 — Run the Air Drawing Board

> **Instructions:**
> - **1 finger up** = Draw mode
> -  **2 fingers up** = Color selection mode (hover over a button)
> - Press **`C`** = Clear canvas
> - Press **`Q`** = Quit

>  Click on the **AirCanvas window** before pressing keyboard shortcuts!

In [ ]:
# -----------------------------------------------
# CONFIGURE MEDIAPIPE HAND LANDMARKER
# -----------------------------------------------

BaseOptions           = mp_python.BaseOptions
HandLandmarker        = mp_vision.HandLandmarker
HandLandmarkerOptions = mp_vision.HandLandmarkerOptions
VisionRunningMode     = mp_vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.7,
    min_tracking_confidence=0.7
)

# -----------------------------------------------
# INITIALIZE STATE
# -----------------------------------------------

cap          = cv2.VideoCapture(0)   # Open webcam
canvas       = None                  # Drawing canvas (created after first frame)
prev_x       = 0
prev_y       = 0
draw_color   = (255, 0, 0)           # Default: Blue (BGR)
frame_ts_ms  = 0                     # Timestamp counter for MediaPipe VIDEO mode

# -----------------------------------------------
# MAIN LOOP
# -----------------------------------------------

with HandLandmarker.create_from_options(options) as landmarker:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        # Mirror the frame (selfie view)
        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape

        # Initialize blank canvas on first frame
        if canvas is None:
            canvas = np.zeros((h, w, 3), dtype=np.uint8)

        # --- Hand Detection ---
        rgb        = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_img     = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        frame_ts_ms += 33
        result     = landmarker.detect_for_video(mp_img, frame_ts_ms)

        # --- Draw UI Buttons ---
        draw_buttons(frame, draw_color)

        # --- Process Hand Landmarks ---
        if result.hand_landmarks:
            lm = result.hand_landmarks[0]

            # Draw hand skeleton
            draw_skeleton(frame, lm, h, w)

            # Index fingertip pixel position
            ix = int(lm[8].x * w)
            iy = int(lm[8].y * h)

            index_up, middle_up = fingers_up(lm, h, w)

            # ✌️ SELECTION MODE — 2 fingers raised
            if index_up and middle_up:
                prev_x, prev_y = 0, 0
                cv2.circle(frame, (ix, iy), 14, (255, 255, 255), 2)  # Cursor ring

                if 20 < iy < 70:  # Finger is over the button area
                    for name, (x1, y1, x2, y2), color in buttons:
                        if x1 < ix < x2:
                            draw_color = (0, 0, 0) if name == "ERASER" else color

            #  DRAWING MODE — Only index finger raised
            elif index_up and not middle_up:
                if iy > 80:  # Only draw below button area
                    if prev_x == 0 and prev_y == 0:
                        prev_x, prev_y = ix, iy  # Start of new stroke

                    thickness = 30 if draw_color == (0, 0, 0) else 8  # Eraser is wider
                    cv2.line(canvas, (prev_x, prev_y), (ix, iy), draw_color, thickness)
                    prev_x, prev_y = ix, iy

                    # Cursor dot at fingertip
                    dot_c = (180, 180, 180) if draw_color == (0,0,0) else draw_color
                    cv2.circle(frame, (ix, iy), thickness // 2, dot_c, cv2.FILLED)
            else:
                prev_x, prev_y = 0, 0  # Reset stroke

        # --- Merge Canvas Drawing onto Camera Frame ---
        gray        = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)
        _, mask     = cv2.threshold(gray, 20, 255, cv2.THRESH_BINARY)
        mask_inv    = cv2.bitwise_not(mask)
        frame_bg    = cv2.bitwise_and(frame,  frame,  mask=mask_inv)
        drawing_fg  = cv2.bitwise_and(canvas, canvas, mask=mask)
        output      = cv2.add(frame_bg, drawing_fg)

        # --- Instruction Bar at Bottom ---
        cv2.rectangle(output, (0, h-38), (w, h), (25, 25, 25), cv2.FILLED)
        cv2.putText(output,
            "  [1 finger] Draw    [2 fingers] Pick color    [C] Clear    [Q] Quit",
            (10, h-12), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (210, 210, 210), 1)

        # --- Display ---
        cv2.imshow("AirCanvas - Virtual Drawing Board", output)

        # --- Keyboard Controls ---
        key = cv2.waitKey(1) & 0xFF
        if key == ord("c"):
            canvas = np.zeros((h, w, 3), dtype=np.uint8)  # Clear canvas
        elif key == ord("q"):
            break

# Cleanup
cap.release()
cv2.destroyAllWindows()
print(" AirCanvas closed.")

 AirCanvas closed.


---

## How It Works — Summary

```
Webcam Frame
     │
     ▼
Flip (mirror) + Convert to RGB
     │
     ▼
MediaPipe HandLandmarker → 21 (x,y,z) landmark points
     │
     ├─ 2 fingers up? ──► SELECTION MODE → hover over buttons to pick color
     │
     ├─ 1 finger up? ───► DRAWING MODE → draw lines on canvas
     │
     └─ no fingers?  ───► IDLE → reset stroke
     │
     ▼
Canvas (drawings) merged with Camera frame via bitwise masking
     │
     ▼
Display final output in OpenCV window
```

